# Multi-Agent Study Assistant with Ollama

## Overview

In this notebook we build a **multi-agent system** using local LLMs.

Instead of one model performing every task, we create specialized agents:

- **Research Agent** – Finds relevant information from the PDF.
- **Writer Agent** – Produces polished study notes.
- **Supervisor** – Coordinates the workflow.

### Workflow

```
                User
                  │
                  ▼
           Supervisor Agent
          ┌────────┴────────┐
          ▼                 ▼
  Research Agent      Writer Agent
          │                 │
          └───────┬───────┘
                   ▼
            study_note.md
```

In [ ]:
from pathlib import Path

from pypdf import PdfReader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np
import ollama

## Load the PDF

In [ ]:
pdf_path = Path("assets/claude_certification_foundation_associate.pdf")

reader = PdfReader(pdf_path)

document = ""

for page in reader.pages:
    text = page.extract_text()

    if text:
        document += text + "\n"

In [ ]:
def chunk_text(text, chunk_size=600, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(document)

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")

vectors = vectorizer.fit_transform(chunks)

In [ ]:
def retrieve(query, top_k=4):

    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, vectors).flatten()

    indices = similarities.argsort()[::-1][:top_k]

    return "\n\n".join(chunks[i] for i in indices)

# Research Agent

Responsibilities

- Read context
- Extract facts
- Never invent information
- Produce research notes

In [ ]:
RESEARCH_SYSTEM = """
You are a research assistant.

Only use the supplied context.

Extract important facts.

Organize them into concise notes.

Do not summarize beyond what is contained in the context.
"""

In [ ]:
def research_agent(question):

    context = retrieve(question)

    prompt = f"""
Context

{context}

Question

{question}
"""

    response = ollama.chat(
        model="qwen3",
        messages=[
            {
                "role": "system",
                "content": RESEARCH_SYSTEM,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )

    return response["message"]["content"]

# Writer Agent

Responsibilities

- Improve readability
- Organize information
- Produce Markdown
- Create headings and bullet lists

In [ ]:
WRITER_SYSTEM = """
You are a technical writer.

Convert research notes into beautiful Markdown.

Use:

# Heading

## Sections

Bullet Lists

Tables if useful.

Keep everything factual.

Never add information that is not present.
"""

In [ ]:
def writer_agent(research_notes):

    response = ollama.chat(
        model="qwen3",
        messages=[
            {
                "role": "system",
                "content": WRITER_SYSTEM,
            },
            {
                "role": "user",
                "content": research_notes,
            },
        ],
    )

    return response["message"]["content"]

# Supervisor

The supervisor coordinates the agents.

Workflow:

User Question

↓

Research Agent

↓

Writer Agent

↓

Markdown File

In [ ]:
def supervisor(question):

    print("Running Research Agent...")

    notes = research_agent(question)

    print("Running Writer Agent...")

    markdown = writer_agent(notes)

    return markdown

In [ ]:
question = """
Create comprehensive study notes for the Claude Certification Foundation Associate PDF.
"""

study_note = supervisor(question)

In [ ]:
print(study_note)

In [ ]:
output = Path("study_note.md")

output.write_text(
    study_note,
    encoding="utf-8"
)

print(f"Saved to {output.resolve()}")

## Example Questions

Try asking:

- Summarize the certification objectives.
- Explain the exam domains.
- What are the prerequisites?
- Produce a one-page study guide.
- Create interview preparation notes.
- Generate a checklist of key concepts.

# Next Steps

Possible enhancements:

- Replace TF-IDF with SentenceTransformers embeddings.
- Store vectors in FAISS or ChromaDB.
- Add conversation memory.
- Introduce a Reviewer Agent to validate outputs.
- Use LangGraph or CrewAI for more advanced orchestration.
- Add web search tools for external knowledge.
- Stream responses from Ollama.
- Build a FastAPI or Streamlit interface.

🎉 Project Complete

At this point, you have a complete educational implementation covering:

Notebook 1: Retrieval-Augmented Generation (RAG) over PDFs using TF-IDF and Ollama.
Notebook 2: A single AI agent that performs tool calling (PDF search, weather lookup, Markdown writing).
Notebook 3: A simple multi-agent workflow with a Research Agent, Writer Agent, and Supervisor coordinating the process.

Suggested Production Enhancements

If you want to evolve this into a modern production-grade agent platform (similar to current enterprise AI systems), consider adding:

LangGraph for stateful agent orchestration.
FAISS, ChromaDB, or Pinecone for semantic vector search.
SentenceTransformers or embedding models from Ollama instead of TF-IDF.
Structured outputs with Pydantic models.
Streaming responses for improved user experience.
Conversation memory (short-term and long-term).
Evaluation pipelines using LangSmith or DeepEval.
Guardrails for prompt injection and hallucination prevention.
Observability with OpenTelemetry, Langfuse, or Arize Phoenix.
A FastAPI or Streamlit frontend.
Containerization with Docker and deployment using Kubernetes or cloud services.

These enhancements will make the project closer to the kinds of agentic AI systems used in production today.